# 03 — Data Cleaning

**Objective:** Transform `raw_observations.csv` into `cleaned_observations.csv` with standardized values, handled missing data, and documented quality decisions.

**Assignment requirement:** *"Explain your design: Why was each field included? How were duplicates and missing values handled? Which information could not be verified?"*

In [ ]:
import pandas as pd
import numpy as np

# Load raw data
raw = pd.read_csv('../data/raw/raw_observations.csv')
print(f"Raw observations loaded: {len(raw)} rows")
print(f"Columns: {list(raw.columns)}")
raw.head()

## 3.1 Data Quality Assessment

Before cleaning, we assess the state of the raw data.

In [ ]:
# Check missing values per column
print("=== Missing Values ===")
missing = raw.isnull().sum()
missing_pct = (raw.isnull().sum() / len(raw) * 100).round(1)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_report[missing_report['missing_count'] > 0])
print()

# Check unique vehicles and part categories
print("=== Unique Vehicles ===")
print(raw['vehicle'].unique())
print()
print("=== Unique Part Categories ===")
print(raw['part_category'].unique())
print()

# Check for potential duplicates
print("=== Potential Duplicates ===")
dupes = raw.duplicated(subset=['vehicle', 'part_category', 'oem_part_number', 'supplier'], keep=False)
print(f"Rows with same vehicle + part + OEM number + supplier: {dupes.sum()}")

## 3.2 Cleaning Steps

### Step 1: Standardize Vehicle Names
Ensure consistent capitalization and naming.

In [ ]:
cleaned = raw.copy()

# Standardize vehicle names
vehicle_mapping = {
    # Add any variations found in raw data
    'byd seagull': 'BYD Seagull',
    'BYD SEAGULL': 'BYD Seagull',
    'Seagull': 'BYD Seagull',
    'byd yuan up': 'BYD Yuan Up',
    'BYD YUAN UP': 'BYD Yuan Up',
    'Yuan Up': 'BYD Yuan Up',
    'toyota bz4x': 'Toyota bZ4X',
    'Toyota BZ4X': 'Toyota bZ4X',
    'TOYOTA BZ4X': 'Toyota bZ4X',
    'bZ4X': 'Toyota bZ4X',
}

cleaned['vehicle'] = cleaned['vehicle'].replace(vehicle_mapping)
print("Standardized vehicles:", cleaned['vehicle'].unique())

### Step 2: Standardize Part Categories

In [ ]:
# Standardize part category names
category_mapping = {
    # Add variations as found in raw data
    'brake pads': 'Brake pads',
    'Brake Pads': 'Brake pads',
    'brake discs': 'Brake discs',
    'Brake Discs': 'Brake discs',
    'brake rotors': 'Brake discs',
    'cabin air filter': 'Cabin-air filter',
    'Cabin Air Filter': 'Cabin-air filter',
    'cabin-air filter': 'Cabin-air filter',
    'wiper blades': 'Wiper blades',
    'Wiper Blades': 'Wiper blades',
    'headlight': 'Headlight assembly',
    'Headlight': 'Headlight assembly',
    'headlight assembly': 'Headlight assembly',
    'tail light': 'Tail-light assembly',
    'Tail Light': 'Tail-light assembly',
    'tail-light assembly': 'Tail-light assembly',
    'side mirror': 'Side mirror',
    'Side Mirror': 'Side mirror',
    'bumper': 'Bumper',
    'fender': 'Fender / body panel',
    'body panel': 'Fender / body panel',
    'fender / body panel': 'Fender / body panel',
    'shock absorber': 'Shock absorber',
    'Shock Absorber': 'Shock absorber',
}

cleaned['part_category'] = cleaned['part_category'].replace(category_mapping)
print("Standardized part categories:", cleaned['part_category'].unique())

### Step 3: Standardize Prices to USD

In [ ]:
# Convert all prices to USD
# Exchange rates as of collection date (2026-11-17)
# Update these with actual rates when you collect data
exchange_rates = {
    'USD': 1.0,
    'CNY': 0.14,   # Chinese Yuan to USD (approximate)
    'ETB': 0.008,  # Ethiopian Birr to USD (approximate)
    'EUR': 1.05,   # Euro to USD (approximate)
}

# Create price_usd column
cleaned['price'] = pd.to_numeric(cleaned['price'], errors='coerce')

def convert_to_usd(row):
    if pd.isna(row['price']) or pd.isna(row.get('currency')):
        return np.nan
    currency = str(row['currency']).upper().strip()
    rate = exchange_rates.get(currency, np.nan)
    if pd.isna(rate):
        return np.nan
    return round(row['price'] * rate, 2)

cleaned['price_usd'] = cleaned.apply(convert_to_usd, axis=1)

# Flag missing prices
cleaned['price_missing'] = cleaned['price_usd'].isna()
print(f"Observations with price: {(~cleaned['price_missing']).sum()}")
print(f"Observations without price: {cleaned['price_missing'].sum()}")

### Step 4: Handle Duplicates and Cross-Checking

**Important:** Two observations of the same part from different sources are NOT duplicates — they are cross-checking evidence. Only flag as a true duplicate when the same source, same supplier, and same part number appear twice.

In [ ]:
# Mark cross-checked observations
# A vehicle-part combination is cross-checked if it has observations from 2+ different sources
source_counts = cleaned.groupby(['vehicle', 'part_category'])['source_id'].nunique().reset_index()
source_counts.columns = ['vehicle', 'part_category', 'source_count']
source_counts['is_cross_checked'] = source_counts['source_count'] >= 2

cleaned = cleaned.merge(
    source_counts[['vehicle', 'part_category', 'is_cross_checked']],
    on=['vehicle', 'part_category'],
    how='left'
)

print(f"Cross-checked combinations: {source_counts['is_cross_checked'].sum()} / {len(source_counts)}")
print(f"Single-source combinations: {(~source_counts['is_cross_checked']).sum()} / {len(source_counts)}")

### Step 5: Classify Compatibility Strength

In [ ]:
# Classify compatibility evidence strength
def classify_compatibility(row):
    statement = str(row.get('compatibility_statement', '')).lower()
    has_oem = pd.notna(row.get('oem_part_number')) and str(row.get('oem_part_number')).strip() != ''
    
    if has_oem and ('confirmed' in statement or 'oem' in statement or 'exact' in statement):
        return 'confirmed_oem'       # OEM part number + explicit compatibility statement
    elif has_oem:
        return 'catalogue_match'      # OEM part number found in catalogue
    elif 'compatible' in statement or 'fits' in statement:
        return 'generic_listing'      # Supplier claims compatibility without OEM number
    elif 'not' in statement:
        return 'not_verifiable'       # Listing exists but compatibility unclear
    else:
        return 'unknown'              # Insufficient information

cleaned['compatibility_strength'] = cleaned.apply(classify_compatibility, axis=1)
print("Compatibility strength distribution:")
print(cleaned['compatibility_strength'].value_counts())

## 3.3 Data Quality Summary

In [ ]:
print("=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)
print(f"Raw observations:                {len(raw)}")
print(f"Cleaned observations:            {len(cleaned)}")
print(f"Observations with price:         {(~cleaned['price_missing']).sum()}")
print(f"Observations missing price:      {cleaned['price_missing'].sum()}")
print(f"Cross-checked combinations:      {cleaned[cleaned['is_cross_checked'] == True]['part_category'].nunique() if 'is_cross_checked' in cleaned.columns else 'N/A'}")
print(f"Unique vehicles:                 {cleaned['vehicle'].nunique()}")
print(f"Unique part categories:          {cleaned['part_category'].nunique()}")
print(f"Vehicle-part combinations:       {cleaned.groupby(['vehicle', 'part_category']).ngroups}")
print("=" * 60)

## 3.4 Field Justification

| Field | Why Included | Type |
|-------|-------------|------|
| `observation_id` | Unique identifier for traceability | ID |
| `vehicle` | Which EV model this observation is about | Category |
| `model_year` | Parts may differ by year; needed for fitment accuracy | Attribute |
| `part_category` | One of the 10 assigned categories | Category |
| `part_description` | Free-text description from the source | Evidence |
| `oem_part_number` | Key for cross-referencing and confirming compatibility | Evidence |
| `supplier` | Who sells/lists this part | Evidence |
| `price_usd` | Standardized price for comparison | Numeric |
| `availability` | Whether the part is in stock / available to order | Evidence |
| `compatibility_statement` | What the source says about fitment | Evidence |
| `compatibility_strength` | Our classification of how reliable the fitment evidence is | Derived |
| `source_id` | Links back to `source_log.csv` for full source details | Traceability |
| `source_url` | Direct URL where the evidence was found | Traceability |
| `collection_date` | When we accessed the source | Traceability |
| `evidence_type` | Type of evidence (catalogue_listing, complaint, recall, etc.) | Classification |
| `is_cross_checked` | Whether another source confirms this vehicle-part combo | Quality |
| `price_missing` | Flag for observations where price could not be found | Quality |
| `notes` | Any additional context or caveats | Documentation |

In [ ]:
# Save cleaned dataset
cleaned.to_csv('../data/cleaned/cleaned_observations.csv', index=False)
print(f"Cleaned dataset saved: {len(cleaned)} rows")
print(f"Output: data/cleaned/cleaned_observations.csv")